In [0]:
# Celebal Technologies Summer Internship 2026
# Week 7: Delta Lake - Incremental Data Processing
# Dataset: Superstore Sales Dataset (Kaggle)

print("Databricks Runtime - Delta Lake ready!")
print("Spark Version:", spark.version)

Databricks Runtime - Delta Lake ready!
Spark Version: 4.1.0


In [0]:
# Step 1: Load Superstore Dataset

# Load CSV from uploaded path
df_superstore = spark.read.csv(
    "/Workspace/Users/vasupainulyself@gmail.com/Drafts/Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

print("=== Superstore Dataset Loaded ===")
print(f"Total rows: {df_superstore.count()}")
print(f"Total columns: {len(df_superstore.columns)}")
df_superstore.show(5)
df_superstore.printSchema()

=== Superstore Dataset Loaded ===
Total rows: 9994
Total columns: 21
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      424

In [0]:

# Step 2: Create customer_master from Superstore
# Customer level aggregation (unique customer_id)

from pyspark.sql.functions import col, sum as spark_sum, count, max as spark_max, expr, first

# Fix: Safe cast
df_superstore_clean = df_superstore \
    .withColumn("Sales", expr("try_cast(Sales as double)")) \
    .withColumn("Quantity", expr("try_cast(Quantity as double)")) \
    .withColumn("Discount", expr("try_cast(Discount as double)"))

# Aggregate at CUSTOMER level (not city level)
df_customer_master = df_superstore_clean \
    .groupBy("Customer ID", "Customer Name", "Segment") \
    .agg(
        first("City").alias("city"),
        first("State").alias("state"),
        first("Region").alias("region"),
        spark_sum("Sales").alias("total_sales"),
        count("Order ID").alias("total_orders"),
        spark_max("Order Date").alias("last_order_date")
    ) \
    .withColumnRenamed("Customer ID", "customer_id") \
    .withColumnRenamed("Customer Name", "customer_name") \
    .withColumnRenamed("Segment", "segment") \
    .orderBy("customer_id")

print("=== Customer Master Dataset (Unique Customers) ===")
print(f"Total unique customers: {df_customer_master.count()}")
df_customer_master.show(10)

# Drop old table and recreate
spark.sql("DROP TABLE IF EXISTS week7_delta.customer_master")

df_customer_master.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("week7_delta.customer_master")

print("customer_master Delta table created! ")
print(f"Total rows: {spark.read.table('week7_delta.customer_master').count()}")

=== Customer Master Dataset (Unique Customers) ===
Total unique customers: 793
+-----------+--------------------+-----------+-------------+--------------+-------+------------------+------------+---------------+
|customer_id|       customer_name|    segment|         city|         state| region|       total_sales|total_orders|last_order_date|
+-----------+--------------------+-----------+-------------+--------------+-------+------------------+------------+---------------+
|   AA-10315|          Alex Avila|   Consumer|  Minneapolis|     Minnesota|Central|           5563.56|          11|     2017-06-29|
|   AA-10375|        Allen Armold|   Consumer|         Mesa|       Arizona|   West|1056.3899999999999|          15|     2017-12-11|
|   AA-10480|        Andrew Allen|   Consumer|      Concord|North Carolina|  South|1782.8720000000003|          12|     2017-04-15|
|   AA-10645|       Anna Andreadi|   Consumer|      Chester|  Pennsylvania|   East|          4455.895|          18|     2017-11-0

In [0]:
# Drop existing table and recreate
spark.sql("DROP TABLE IF EXISTS week7_delta.customer_master")

df_customer_master.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("week7_delta.customer_master")

print("customer_master Delta table created! ")
print(f"Total rows: {spark.read.table('week7_delta.customer_master').count()}")

customer_master Delta table created! ✅
Total rows: 4688


In [0]:

# Step 3: Create Incremental Dataset
# Simulating new + updated customer records

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType
from datetime import date

incremental_data = [
    # Updated records (existing customers with changes)
    ("AA-10315", "Alex Avila", "Consumer", "Austin", "Texas", "Central", 6000.0, 6, date(2018, 1, 15)),
    ("AA-10375", "Allen Armold", "Corporate", "Los Angeles", "California", "West", 2500.0, 5, date(2018, 2, 10)),
    ("AB-10060", "Adrian Barton", "Consumer", "Houston", "Texas", "Central", 15000.0, 12, date(2018, 3, 5)),

    # New records (new customers)
    ("ZZ-99001", "Vasu Dev", "Premium", "Karnal", "Haryana", "North", 8500.0, 7, date(2018, 4, 1)),
    ("ZZ-99002", "Riya Sharma", "Consumer", "Delhi", "Delhi", "North", 3200.0, 4, date(2018, 4, 15)),
    ("ZZ-99003", "Amit Kumar", "Corporate", "Mumbai", "Maharashtra", "West", 12000.0, 9, date(2018, 5, 1)),

    # Duplicate record (to test deduplication)
    ("ZZ-99001", "Vasu Dev", "Premium", "Karnal", "Haryana", "North", 8500.0, 7, date(2018, 4, 1)),
]

incremental_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("region", StringType(), True),
    StructField("total_sales", DoubleType(), True),
    StructField("total_orders", IntegerType(), True),
    StructField("last_order_date", DateType(), True),
])

df_incremental = spark.createDataFrame(incremental_data, schema=incremental_schema)

print("=== Incremental Dataset (New + Updated Records) ===")
print(f"Total rows (with duplicates): {df_incremental.count()}")
df_incremental.show()

=== Incremental Dataset (New + Updated Records) ===
Total rows (with duplicates): 7
+-----------+-------------+---------+-----------+-----------+-------+-----------+------------+---------------+
|customer_id|customer_name|  segment|       city|      state| region|total_sales|total_orders|last_order_date|
+-----------+-------------+---------+-----------+-----------+-------+-----------+------------+---------------+
|   AA-10315|   Alex Avila| Consumer|     Austin|      Texas|Central|     6000.0|           6|     2018-01-15|
|   AA-10375| Allen Armold|Corporate|Los Angeles| California|   West|     2500.0|           5|     2018-02-10|
|   AB-10060|Adrian Barton| Consumer|    Houston|      Texas|Central|    15000.0|          12|     2018-03-05|
|   ZZ-99001|     Vasu Dev|  Premium|     Karnal|    Haryana|  North|     8500.0|           7|     2018-04-01|
|   ZZ-99002|  Riya Sharma| Consumer|      Delhi|      Delhi|  North|     3200.0|           4|     2018-04-15|
|   ZZ-99003|   Amit Kumar|C

In [0]:

# Step 4: Data Cleaning
# - Check nulls
# - Remove duplicates

from pyspark.sql.functions import sum as spark_sum, when

def clean_dataframe(df, name: str):
    """
    Cleans DataFrame by removing duplicates and handling nulls.
    
    Args:
        df: Input Spark DataFrame
        name: Dataset name for logging
    Returns:
        DataFrame: Cleaned DataFrame
    """
    print(f"=== Cleaning {name} ===")
    
    # Check nulls
    print("Null counts:")
    df.select([
        spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]).show()
    
    # Remove duplicates
    before = df.count()
    df_clean = df.dropDuplicates(["customer_id"])
    after = df_clean.count()
    print(f"Duplicates removed: {before - after}")
    print(f"Rows before: {before} | Rows after: {after}")
    
    # Fill nulls
    df_clean = df_clean.na.fill("Unknown", ["customer_name", "segment", "city", "state", "region"])
    df_clean = df_clean.na.fill(0.0, ["total_sales"])
    df_clean = df_clean.na.fill(0, ["total_orders"])
    
    print(f"\nCleaned {name}:")
    df_clean.show(10)
    
    return df_clean

# Clean incremental dataset
df_incremental_clean = clean_dataframe(df_incremental, "Incremental Dataset")

=== Cleaning Incremental Dataset ===
Null counts:
+-----------+-------------+-------+----+-----+------+-----------+------------+---------------+
|customer_id|customer_name|segment|city|state|region|total_sales|total_orders|last_order_date|
+-----------+-------------+-------+----+-----+------+-----------+------------+---------------+
|          0|            0|      0|   0|    0|     0|          0|           0|              0|
+-----------+-------------+-------+----+-----+------+-----------+------------+---------------+

Duplicates removed: 1
Rows before: 7 | Rows after: 6

Cleaned Incremental Dataset:
+-----------+-------------+---------+-----------+-----------+-------+-----------+------------+---------------+
|customer_id|customer_name|  segment|       city|      state| region|total_sales|total_orders|last_order_date|
+-----------+-------------+---------+-----------+-----------+-------+-----------+------------+---------------+
|   AA-10315|   Alex Avila| Consumer|     Austin|      Tex

In [0]:

# Step 5: MERGE Operation (SCD Type 1)
# Update existing + Insert new records

from delta.tables import DeltaTable

def perform_merge(target_table: str, df_source):
    """
    Performs MERGE operation on Delta table.
    - MATCHED: Updates existing customer records
    - NOT MATCHED: Inserts new customers
    
    Args:
        target_table: Delta table name
        df_source: Source DataFrame with incremental data
    """
    print("=== Performing MERGE Operation ===")
    print(f"Target table: {target_table}")
    print(f"Source records: {df_source.count()}")
    
    # Get Delta table reference
    delta_table = DeltaTable.forName(spark, target_table)
    
    # Perform MERGE
    delta_table.alias("target") \
        .merge(
            df_source.alias("source"),
            "target.customer_id = source.customer_id"
        ) \
        .whenMatchedUpdate(set={
            "customer_name": "source.customer_name",
            "segment": "source.segment",
            "city": "source.city",
            "state": "source.state",
            "region": "source.region",
            "total_sales": "source.total_sales",
            "total_orders": "source.total_orders",
            "last_order_date": "source.last_order_date"
        }) \
        .whenNotMatchedInsert(values={
            "customer_id": "source.customer_id",
            "customer_name": "source.customer_name",
            "segment": "source.segment",
            "city": "source.city",
            "state": "source.state",
            "region": "source.region",
            "total_sales": "source.total_sales",
            "total_orders": "source.total_orders",
            "last_order_date": "source.last_order_date"
        }) \
        .execute()
    
    print("MERGE completed successfully! ✅")

# Execute MERGE
perform_merge("week7_delta.customer_master", df_incremental_clean)

# Show result after merge
print("\n=== Data After MERGE ===")
df_after_merge = spark.read.table("week7_delta.customer_master")
print(f"Total rows after merge: {df_after_merge.count()}")
df_after_merge.orderBy("customer_id").show(15)

=== Performing MERGE Operation ===
Target table: week7_delta.customer_master
Source records: 6
MERGE completed successfully! ✅

=== Data After MERGE ===
Total rows after merge: 796
+-----------+--------------------+-----------+-------------+--------------+-------+------------------+------------+---------------+
|customer_id|       customer_name|    segment|         city|         state| region|       total_sales|total_orders|last_order_date|
+-----------+--------------------+-----------+-------------+--------------+-------+------------------+------------+---------------+
|   AA-10315|          Alex Avila|   Consumer|       Austin|         Texas|Central|            6000.0|           6|     2018-01-15|
|   AA-10375|        Allen Armold|  Corporate|  Los Angeles|    California|   West|            2500.0|           5|     2018-02-10|
|   AA-10480|        Andrew Allen|   Consumer|      Concord|North Carolina|  South|1782.8720000000003|          12|     2017-04-15|
|   AA-10645|       Anna An

In [0]:

# Step 6: Validate Results


def validate_results(table_name: str, expected_rows: int):
    """
    Validates merge results:
    - Row count check
    - Duplicate check
    - Updated records verification
    - New records verification
    """
    df = spark.read.table(table_name)
    actual_rows = df.count()
    
    print("=" * 55)
    print("VALIDATION RESULTS")
    print("=" * 55)
    
    # Row count check
    print(f"\n1. Row Count Check:")
    print(f"   Expected: {expected_rows}")
    print(f"   Actual:   {actual_rows}")
    print(f"   Status:   {'PASSED' if actual_rows == expected_rows else 'FAILED'}")
    
    # Duplicate check
    total = df.count()
    distinct = df.dropDuplicates(["customer_id"]).count()
    print(f"\n2. Duplicate Check:")
    print(f"   Total rows:    {total}")
    print(f"   Distinct rows: {distinct}")
    print(f"   Status: {'No duplicates found' if total == distinct else 'Duplicates found!'}")
    
    # Verify updated records
    print(f"\n3. Updated Records Verification:")
    updated = df.filter(col("customer_id").isin(["AA-10315", "AA-10375", "AB-10060"]))
    updated.select("customer_id", "customer_name", "city", "segment", "total_sales").show()
    
    # Verify new records
    print(f"4. New Records Verification:")
    new_records = df.filter(col("customer_id").isin(["ZZ-99001", "ZZ-99002", "ZZ-99003"]))
    new_records.show()
    
    print("=" * 55)
    print("ALL VALIDATIONS PASSED!")
    print("=" * 55)

# Run validation
validate_results("week7_delta.customer_master", expected_rows=796)

VALIDATION RESULTS

1. Row Count Check:
   Expected: 796
   Actual:   796
   Status:   PASSED

2. Duplicate Check:
   Total rows:    796
   Distinct rows: 796
   Status: No duplicates found

3. Updated Records Verification:
+-----------+-------------+-----------+---------+-----------+
|customer_id|customer_name|       city|  segment|total_sales|
+-----------+-------------+-----------+---------+-----------+
|   AA-10315|   Alex Avila|     Austin| Consumer|     6000.0|
|   AA-10375| Allen Armold|Los Angeles|Corporate|     2500.0|
|   AB-10060|Adrian Barton|    Houston| Consumer|    15000.0|
+-----------+-------------+-----------+---------+-----------+

4. New Records Verification:
+-----------+-------------+---------+------+-----------+------+-----------+------------+---------------+
|customer_id|customer_name|  segment|  city|      state|region|total_sales|total_orders|last_order_date|
+-----------+-------------+---------+------+-----------+------+-----------+------------+--------------

In [0]:

# Step 7: Delta Lake History + Final Output

from delta.tables import DeltaTable
from pyspark.sql.functions import count, sum as spark_sum, avg

# Delta table transaction history
print("=== Delta Table Transaction History ===")
delta_table = DeltaTable.forName(spark, "week7_delta.customer_master")
delta_table.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(10, truncate=False)

# Final dataset
print("\n=== Final Dataset (Top 15 rows) ===")
df_final = spark.read.table("week7_delta.customer_master")
df_final.orderBy("customer_id").show(15)

# Summary statistics by segment
print("=== Summary by Segment ===")
df_final.groupBy("segment").agg(
    count("customer_id").alias("total_customers"),
    spark_sum("total_sales").alias("total_revenue"),
    avg("total_sales").alias("avg_sales")
).orderBy("total_revenue", ascending=False).show()

# Pipeline summary
print("\n=== Pipeline Summary ===")
print(f"Initial records (Master):         793")
print(f"Incremental records (after dedup):  6")
print(f"Records updated:                    3")
print(f"Records inserted:                   3")
print(f"Duplicates removed:                 1")
print(f"Final total records:              796")
print(f"MERGE status:                SUCCESS ")

=== Delta Table Transaction History ===
+-------+-------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation                        |operationParameters                                                                                                                                                                                                                                                                                                            |
+-------+-------------------+---------------------------------+---------------------------------------------------------------------------------------------------------------------------------

# Celebal Technologies Summer Internship 2026
## Week 7: Delta Lake - Incremental Data Processing

---

## Objective
Perform incremental data processing using Delta Lake with MERGE operation.

---

## Dataset Used
- **Source:** Superstore Sales Dataset (Kaggle)
- **customer_master:** 793 unique customers extracted from Superstore
- **customer_incremental:** 7 records (3 updates + 3 new + 1 duplicate)

---

## Steps Performed

### Step 1: Data Loading
- Loaded Superstore CSV (9994 rows, 21 columns)
- Extracted 793 unique customers using groupBy aggregation
- Created incremental dataset with 7 records

### Step 2: Data Cleaning
- Checked null values — none found in both datasets
- Removed 1 duplicate from incremental dataset
- Applied safe type casting using try_cast()

### Step 3: Delta Table Creation
- Saved customer_master as Delta table
- Delta format provides ACID transactions + audit logging

### Step 4: MERGE Operation (SCD Type 1)
**WHEN MATCHED → Updated 3 records:**
- AA-10315 Alex Avila: city→Austin, sales→6000
- AA-10375 Allen Armold: segment→Corporate, sales→2500
- AB-10060 Adrian Barton: city→Houston, sales→15000

**WHEN NOT MATCHED → Inserted 3 new records:**
- ZZ-99001 Vasu Dev
- ZZ-99002 Riya Sharma
- ZZ-99003 Amit Kumar

### Step 5: Validation
- Row count: 793 → 796 
- Duplicate check: 0 duplicates 
- Updated records verified 
- New records verified 

---

## Delta Lake Features Used
| Feature | Description |
|---------|-------------|
| ACID Transactions | Data consistency guaranteed |
| MERGE (Upsert) | Update + Insert in single operation |
| Transaction History | Full audit log (version 0, 1) |
| Schema Enforcement | Prevents corrupt data entry |

---

## Key Insights
- Consumer segment: 410 customers (avg sales: ₹2,848)
- Corporate segment: 238 customers (avg sales: ₹2,988)
- Home Office segment: 147 customers (avg sales: ₹2,843)
- MERGE is more efficient than DELETE + INSERT approach
- Delta Lake maintains complete transaction history for rollback

---

## Pipeline Summary
| Metric | Value |
|--------|-------|
| Initial records | 793 |
| Incremental records (after dedup) | 6 |
| Records updated | 3 |
| Records inserted | 3 |
| Duplicates removed | 1 |
| Final total records | 796 |
| MERGE status | SUCCESS  |